<h2><b><center><span style="color:blue">LANGGRAPH

- LangGraph is an <span style="color: limegreen;">orchestration framework</span> for applications that need multiple steps, shared state, branching, loops, and recovery. It lets you combine predictable Python logic with LLM-driven decisions.
- LangGraph helps you build an application as connected steps, carry data between those steps, and recover when execution fails.
<br>

These notes cover its main concepts and production applications. We’ll use one example throughout: read an email, extract funding details, validate them, and save the result.

### <span style="color: magenta;">1. Why do we need LangGraph?</span>

A simple AI application sends a prompt to an LLM and receives an answer. A larger application might need to:

- Extract information.
- Look up related records.
- Run independent tasks together.
- Correct an invalid result.
- Wait for a reviewer.
- Recover after a crash.

LangGraph helps coordinate these operations. You still write the prompts, functions, business rules, and validation.

### <span style="color: magenta;">2. Where it fits in the AI stack</span>

| <span style="color: lightblue;">Component       | <span style="color: lightblue;">Responsibility                                             | <span style="color: lightblue;">Simple example                    |
| --------------- | ---------------------------------------------------------- | --------------------------------- |
| <span style="color: limegreen;">**LLM**         | Understand language and produce responses                  | Extract CAD 5,000 from an email   |
| <span style="color: limegreen;">**LangChain**   | Provide model/tool integrations and agent abstractions     | Connect an agent to a search tool |
| <span style="color: limegreen;">**LangGraph**   | Manage execution, state, and recovery                      | Extract → validate → save         |
| <span style="color: limegreen;">**Deep Agents** | Provide planning, subagents, files, and context management | Coordinate a longer research task |
| <span style="color: limegreen;">**LangSmith**   | Trace, evaluate, debug, and deploy applications            | Inspect why a workflow failed     |

<br>

We can call existing model API client inside LangGraph nodes. LangChain model wrappers and LangSmith are optional for using the core framework.

### <span style="color: magenta;">3. What graph and state machine mean</span>

- A graph contains steps called <span style="color: limegreen;">nodes</span>, connected by paths called <span style="color: limegreen;">edges</span>.
- <span style="color: limegreen;">A directed graph</span> specifies which direction execution can move.
- A <span style="color: limegreen;">state machine</span> chooses actions according to its current data and rules.
- LangGraph combines these ideas: <span style="color: limegreen;">execute a step</span>, <span style="color: limegreen;">update data</span>, and <span style="color: limegreen;">decide what happens next</span>.
- A DAG cannot loop back. LangGraph <span style="color: limegreen;">supports cycles</span>, allowing repeated correction or tool use.

<span style="color: lightblue;">Example</span>: if validation fails, return to correction; if it passes, continue to saving.

### <span style="color: magenta;">4. The core building blocks</span>

| <span style="color: lightblue;">Concept              | <span style="color: lightblue;">Meaning                                 | <span style="color: lightblue;">Our example                    |
| -------------------- | --------------------------------------- | ------------------------------ |
| <span style="color: limegreen;">**State**            | Working data available during execution | Email, amounts, errors         |
| <span style="color: limegreen;">**Node**             | Function that performs an operation     | Extract amounts                |
| <span style="color: limegreen;">**Edge**             | Connection between steps                | Extract → validate             |
| <span style="color: limegreen;">**Conditional edge** | Connection selected using a condition   | Valid → save; unclear → review |
| <span style="color: limegreen;">**START**            | Entry marker                            | Begin processing               |
| <span style="color: limegreen;">**END**              | Completion marker                       | Finish processing              |


<br>

A node can execute Python, SQL, regex, an API call, or an LLM call. A node does not have to be an agent.

### <span style="color: magenta;">5. State: the workflow’s working data</span>


Imagine passing a folder between workers. Each worker reads what they need and adds their result. State serves a similar purpose.

```python
{
    "email": "We agree to CAD 5,000 for Christmas.",
    "amount": 5000,
    "period": "Christmas",
    "validation_errors": []
}
```

- State changes as nodes complete.
- Nodes normally return <span style="color: limegreen;">partial updates</span>, such as {"amount": 5000}.
- Fields not updated remain available.
- <span style="color: limegreen;">Return updates</span> instead of directly modifying the input state.
- State does not automatically become model input. Your code chooses what to send to the LLM.

### <span style="color: magenta;">6. Schemas: defining the data contract</span>

A schema describes expected fields and types.

| <span style="color: lightblue;">Schema option | <span style="color: lightblue;">Meaning                     |
| ------------- | -------------------------------------------------------------------- |
| <span style="color: limegreen;">`TypedDict`</span>   | Describes dictionary fields for type checking; no runtime validation |
| <span style="color: limegreen;">Dataclass</span>     | Defines structured Python data and defaults                          |
| <span style="color: limegreen;">Pydantic</span>      | Supports validation and conversion of data at runtime                |

There are also different schema roles:

- <span style="color: limegreen;">Input schema</span>: what the caller supplies.
- <span style="color: limegreen;">Internal state schema</span>: information used during execution.
- <span style="color: limegreen;">Output schema</span>: what the workflow returns.
- <span style="color: limegreen;">LLM-output schema</span>: structure expected from a model response.

<span style="color: lightblue;">Example</span>: the graph receives an email, stores intermediate errors, and returns extracted records. The model’s separate output schema requires an amount and currency.

Valid structure does not guarantee correct facts. 5000 can be a valid number while being the wrong amount.


### <span style="color: magenta;">7. Nodes: deciding the size of each step</span>

Give each node a meaningful responsibility.

- <span style="color: limegreen;">Good boundary</span>: extract amounts, validate evidence, save records.
- <span style="color: limegreen;">Too large</span>: one node reads data, makes five calls, validates, and writes everything.
- <span style="color: limegreen;">Too fragmented</span>: separate nodes for every tiny Python operation.

Smaller meaningful nodes make failures easier to isolate and reduce repeated work during recovery. Larger nodes may simplify code but hide intermediate progress.

- A function named `AmountAgent` is still just an extraction function unless it independently chooses actions.
- A function’s name does not make it an agent. An extraction function performs a predefined task, such as extracting amounts from an email. An <span style="color: limegreen;">agent lets the model choose its next action</span>, such as reading an attachment or retrieving another message, using permitted tools.
- Multiple LLM calls executed in a code-defined sequence form an LLM-powered workflow, not necessarily an autonomous agent.

### <span style="color: magenta;">8. Reducers: how updates are combined</span>

A reducer decides how an existing state value combines with a new update.

| <span style="color: lightblue;">Update rule</span>         | <span style="color: lightblue;">Existing value</span> | <span style="color: lightblue;">New update</span> | <span style="color: lightblue;">Result</span>         |
| ------------------- | -------------- | ---------- | -------------- |
| <span style="color: limegreen;">Default replacement</span> | `[5000]`       | `[6000]`   | `[6000]`       |
| <span style="color: limegreen;">List concatenation</span>  | `[5000]`       | `[6000]`   | `[5000, 6000]` |

- `operator.add` concatenates lists; it does not remove duplicates.
- With an append reducer, return only the new items.
- An empty list does not clear an append field because adding nothing preserves its contents.
- `Overwrite` explicitly replaces a reducer-managed value.

<span style="color: lightblue;">Example</span>: For revised amounts, replacement is usually appropriate. For results from different attachments, accumulation may be appropriate.

### <span style="color: magenta;">9. Messages and conversation history</span>

Agents often store exchanges as messages:

- <span style="color: limegreen;">Human message</span>: user input.
- <span style="color: limegreen;">AI message</span>: model response or tool-call request.
- <span style="color: limegreen;">Tool message</span>: result of a tool execution.

`MessagesState` supplies a messages field with the `add_messages` reducer. It <span style="color: limegreen;">appends new messages</span> and <span style="color: limegreen;">updates existing messages</span> by ID.

Saving history does not mean the model automatically sees it. The model node must include the relevant messages in its request.

### <span style="color: magenta;">10. Edges and routing</span>

- A normal edge always goes to a specified next node.
- A <span style="color: limegreen;">conditional edge</span> calls a <span style="color: limegreen;">routing function</span> to select the next node.
- Command can combine a state update and routing decision in one return value.

<span style="color: lightblue;">Example</span>:
```python
if validation_errors:
    next_step = "review"
else:
    next_step = "save"
```

Use Python for clear rules. Use an LLM when routing requires interpreting language.<br>
Avoid accidentally adding both a fixed route and dynamic routing from the same node: <span style="color: limegreen;">both destinations can execute</span>.

### <span style="color: magenta;">11. Parallel execution and supersteps</span>

- <span style="color: limegreen;">Parallel execution</span>: independent nodes run together.
- <span style="color: limegreen;">Fan-out</span>: start several branches.
- <span style="color: limegreen;">Fan-in</span>: combine their results.
- <span style="color: limegreen;">Superstep</span>: one execution round containing the nodes scheduled together.

<span style="color: lightblue;">Example</span>: deal-ID extraction and CBM extraction can run in parallel. A later node combines their outputs.

Nodes in the same superstep work from the state available at its start. One cannot depend on another’s unfinished output.

If parallel nodes update the same field, use a suitable reducer or separate fields. Explicitly join branches when the next step needs all results.

### <span style="color: magenta;">12. Loops, limits, and dynamic tasks</span>

A loop repeats steps until a condition is satisfied.
<span style="color: lighblue;">Example</span>: generate a query → validate it → correct it → validate again.

- Define a stopping condition.
- Limit correction attempts.
- Set time and cost budgets.
- `recursion_limit` limits graph supersteps, not Python recursion or the total number of model calls.

Send creates tasks dynamically. For ten attachments, create ten processing tasks.

This <span style="color: limegreen;">supports map-reduce</span>: process each item independently, then combine the outputs. Parallel work still consumes API capacity and tokens.

### <span style="color: magenta;">13. Workflow versus agent</span>

| <span style="color: lightblue;">Workflow</span>                                | <span style="color: lightblue;">Agent</span>                                  |
| --------------------------------------- | --------------------------------------------------- |
| Your code largely controls the sequence | The model dynamically chooses actions               |
| Usually easier to predict and test      | More flexible for open-ended tasks                  |
| Example: extract → validate → save      | Example: investigate a problem using selected tools |

A typical tool-calling agent repeats:
1. Model proposes a tool call.
2. Application executes the tool.
3. Tool result returns to the model.
4. Model chooses another action or finishes.

The model proposes actions; your application controls whether and how they execute. LangGraph supports workflows, agents, and combinations of both.


### <span style="color: magenta;">14. Common architecture patterns</span>

| <span style="color: lightblue;">Pattern</span>             | <span style="color: lightblue;">Explanation</span>                                         | <span style="color: lightblue;">Example</span>                           |
| ------------------- | --------------------------------------------------- | --------------------------------- |
| <span style="color: limegreen;">Sequential chaining</span> | Each step feeds the next                            | Extract → normalise → validate    |
| <span style="color: limegreen;">Routing</span>             | Select a path or specialist                         | Invoice → invoice processor       |
| <span style="color: limegreen;">Parallelisation</span>     | Run independent operations together                 | Extract IDs and contacts          |
| <span style="color: limegreen;">Orchestrator–worker</span> | Coordinator distributes dynamically identified work | Analyse each section of a report  |
| <span style="color: limegreen;">Evaluator–optimizer</span> | Evaluate an output and request corrections          | Fix a query that fails validation |
| <span style="color: limegreen;">Tool-calling agent</span>  | Model repeatedly selects tools                      | Investigate an unfamiliar issue   |

Choose the simplest pattern that meets the requirement. Multiple agents do not automatically improve quality.


### <span style="color: magenta;">15. Subgraphs and multiple agents</span>

A subgraph is a <span style="color: limegreen;">complete smaller workflow</span> used inside a larger one.

<span style="color: lightblue;">Example</span>: a document-processing graph contains a reusable validation subgraph.

- Parent and subgraph can share state fields.
- If their schemas differ, translate inputs and outputs through an adapter.
- Choose whether subgraph history should persist between calls or remain isolated.

Common multi-agent arrangements:
- <span style="color: limegreen;">Supervisor</span>: a coordinator delegates to specialists.
- <span style="color: limegreen;">Handoff</span>: one specialist transfers responsibility to another.

Use multiple agents when they need distinct tools, instructions, or context. Account for coordination mistakes, extra latency, and cost.

### <span style="color: magenta;">16. Checkpoints, threads, and runs</span>

| <span style="color: lightblue;">Term</span>             | <span style="color: lightblue;">Meaning</span>                                             |
| ---------------- | --------------------------------------------------- |
| <span style="color: limegreen;">**Run**</span>          | An execution attempt                                |
| <span style="color: limegreen;">**Thread**</span>       | Saved execution context that can span multiple runs |
| <span style="color: limegreen;">**Checkpoint**</span>   | Snapshot of state and execution progress            |
| <span style="color: limegreen;">**Checkpointer**</span> | Component that stores and retrieves checkpoints     |

Supply a `thread_id` when using checkpointing.

<span style="color: lightblue;">Example</span>: one ID identifies processing for email A. If processing fails, continue using that same saved context. Email B should have its own context.

Checkpoints capture <span style="color: limegreen;">superstep progress</span>. Saved pending writes can preserve successful parallel results when another branch fails.

In-memory checkpoints disappear after restart. Production recovery requires durable storage.

### <span style="color: magenta;">17. Memory and context engineering</span>

- <span style="color: limegreen;">Short-term memory</span>: data belonging to one thread, such as its conversation.
- <span style="color: limegreen;">Long-term memory</span>: data stored across threads, such as preferences or shared facts.
- A <span style="color: limegreen;">store</span> holds cross-thread data; nodes retrieve it when needed.
- <span style="color: limegreen;">Context engineering</span>: deciding which instructions, evidence, history, and tool results each model call receives.

<span style="color: lightblue;">Example</span>: the status node receives the email plus linked amounts and periods. It does not need unrelated logs.

Longer context is not automatically better. Select relevant information and preserve exact evidence when amounts, IDs, or wording matter.

### <span style="color: magenta;">18. Durable execution and idempotency</span>

- Durable execution allows continuation from persisted progress after interruption or failure. It does not preserve a running Python process at every line.
- <span style="color: limegreen;">Checkpointed tasks</span> can <span style="color: limegreen;">save completed operations</span> within a larger function, reducing repeated work.
- However, an operation may still execute again. <span style="color: limegreen;">Idempotency means repetition does not create duplicate effects</span>.

<span style="color: lightblue;">Example</span>: upsert a record using a stable key instead of blindly appending it.

| <span style="color: lightblue;">Durability mode | <span style="color: lightblue;">When state is persisted   | <span style="color: lightblue;">Main trade-off                               |
| --------------- | ------------------------- | -------------------------------------------- |
| `sync`          | Before the next step      | Stronger durability, more overhead           |
| `async`         | While execution continues | Better performance, a crash window           |
| `exit`          | When execution exits      | Intermediate progress may be lost on a crash |

Checkpointing <span style="color: limegreen;">does not guarantee</span> exactly-once external writes.

### <span style="color: magenta;">19. Errors, retries, and correction</span>

| <span style="color: lightblue;">Problem</span>                   | <span style="color: lightblue;">Appropriate response</span>            |
| ------------------------- | ------------------------------- |
| Temporary network failure | Bounded retry                   |
| Rate limit                | Retry with backoff              |
| Invalid credentials       | Fix configuration               |
| Wrong extraction          | Validate and correct or review  |
| Missing evidence          | Leave unresolved or seek review |

- <span style="color: limegreen;">Backoff</span>: increase the delay between retries.
- <span style="color: limegreen;">Jitter</span>: vary delays so many requests do not retry simultaneously.
- Set API and node timeouts.
- Coordinate SDK, graph, and job retries to avoid multiplying attempts.
- A successful API response can still contain an incorrect answer.
- Record failures clearly instead of silently returning empty results as success.

### <span style="color: magenta;">20. Human review and interrupts</span>

`interrupt()` pauses execution and requests external input.

<span style="color: lightblue;">Example</span>: two periods could apply to one amount. A reviewer selects the supported association.

- Checkpointing preserves the paused context.
- Resume with `Command(resume=...)` and the <span style="color: limegreen;">same thread ID</span>.
- The interrupted node starts again from its beginning.
- Code before the interrupt may run again, so protect side effects.
- Keep interrupt order consistent and avoid swallowing interrupts with broad exception handling.

A reviewer should see the evidence and proposed decision, not just an unexplained approve button.

### <span style="color: magenta;">21. Inspection, replay, and time travel</span>

- `get_state()` inspects current state.
- `get_state_history()` retrieves checkpoint history.
- `update_state()` creates an updated checkpoint using reducer rules.
- `Replay`: continue from an earlier checkpoint.
- `Fork`: change saved state and explore another execution path.

<span style="color: lightblue;">Example</span>: inspect where an incorrect period appeared, change it, and rerun downstream processing.

Downstream LLM calls or writes may execute again. Time travel does not undo an email already sent or a database record already written.

### <span style="color: magenta;">22. Building, running, and streaming</span>

| <span style="color: lightblue;">API</span>                        | <span style="color: lightblue;">Purpose</span>                                      |
| -------------------------- | -------------------------------------------- |
| `StateGraph`               | Build an explicit graph                      |
| `.compile()`               | Prepare it for execution and check structure |
| `.invoke()`                | Run synchronously                            |
| `.ainvoke()`               | Run asynchronously                           |
| `.stream()` / `.astream()` | Receive incremental progress                 |

The Graph API <span style="color: limegreen;">explicitly declares nodes and connections</span>.

The Functional API uses `@entrypoint` and `@task`, retaining ordinary Python control flow while adding persistence and other capabilities.

Streaming can expose state changes, snapshots, model tokens, or custom progress events. Token streaming needs compatible model integration.

Filter streamed data: internal fields can still appear in streams. Compilation checks structure, not business correctness.

### <span style="color: magenta;">23. Runtime context, caching, and performance</span>

- <span style="color: limegreen;">Runtime context</span>: dependencies/configuration used during execution, such as model choice, client identity, or an API client.
- <span style="color: limegreen;">State</span>: changing workflow data.
- <span style="color: limegreen;">Caching</span>: reuse a result instead of repeating computation.
- <span style="color: limegreen;">Checkpointing</span>: save progress so execution can continue.

<span style="color: lightblue;">Example</span>: cache a document lookup, but checkpoint the extraction workflow.

Cache keys must distinguish relevant data, model/prompt versions, and tenants. Set <span style="color: limegreen;">expiry</span> when results can become stale.

Use <span style="color: limegreen;">asynchronous I/O</span> for network-heavy operations and bounded concurrency for throughput. Measure latency and cost per successful result.

### <span style="color: magenta;">24. Testing, observability, and evaluation</span>

- <span style="color: limegreen;">Testing</span>: verify nodes, routes, loops, joins, failures, and recovery.
- <span style="color: limegreen;">Observability</span>: inspect what happened during execution.
- <span style="color: limegreen;">Evaluation</span>: measure whether outputs are useful and correct.

Track node failures, timing, routing, model usage, and tool results.

Use labelled examples to measure extraction precision, recall, and entity-linking accuracy. Test duplicate writes, restart recovery, and human-review continuation.

<span style="color: limegreen;">LangSmith</span> supports tracing and evaluation; MLflow also integrates with LangChain/LangGraph.

An LLM judge can assist evaluation, but exact amounts, IDs, and business rules need direct checks.

### <span style="color: magenta;">25. Deployment, security, and versioning</span>

LangGraph is only part of a production system.

- Run it inside your own service/job or use deployment infrastructure such as Agent Server.
- Manage queues, cancellation, connection pools, and API limits.
- Control concurrent operations on the same thread.
- Authenticate users and isolate tenant data. A thread ID is not permission to access its contents.
- Treat emails and retrieved documents as untrusted input.
- Restrict tools and validate proposed actions before executing them.
- Protect checkpoints and traces with access controls and retention rules.
- Pin versions and plan migrations. Changes to node names or state can break paused executions.

### <span style="color: magenta;">26. Applications and production priorities</span>

| <span style="color: lightblue;">Application</span>               | <span style="color: lightblue;">Why LangGraph can help</span>                                     |
| ------------------------- | ---------------------------------------------------------- |
| Email/document extraction | Dependencies, validation, review, recovery                 |
| RAG                       | Retrieve evidence, evaluate it, and route searches         |
| SQL assistant             | Inspect schemas, validate queries, attempt bounded repairs |
| Customer support          | Preserve conversations and route/escalate cases            |
| Research                  | Distribute tasks, use tools, combine findings              |
| Business automation       | Coordinate persistent steps and approvals                  |
| Coding/data analysis      | Execute tools and iterate on results                       |

These patterns appear in the official application tutorials.

For production work, prioritise predictable behavior, evidence, recovery, evaluation, security, latency, and cost. Add autonomy when it solves a real requirement.

<span style="color: limegreen;">The distinction to remember</span>: LangGraph can make execution reliable, but accurate extraction still depends on your model, prompts, evidence rules, and validation.